# Benchmarking runtimes for TMDD model

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display


%load_ext autoreload
%autoreload 2

from vpop_calibration import *


In [ ]:
def load_data(file: str) -> pd.DataFrame:
    df = pd.read_csv(file)
    return df

In [ ]:
# Make sure to run  `generate_synthetic_data.R` first
train_df = load_data("./data/gp_training.csv")
obs_data = {}
# Make sure this is aligned with the available observed data
patient_counts = [100, 200, 300, 400, 500, 1000, 2000, 5000]
for nb_patients in patient_counts:
    df = load_data(f"./data/obs_data_{nb_patients:d}pts.csv")
    obs_df = df[["id", "time", "DV", "arm"]]
    obs_df = obs_df.loc[~obs_df["DV"].isna()]
    obs_df = obs_df.rename(columns={"DV": "value", "arm": "protocol_arm"})
    obs_df["output_name"] = "DV"
    patients_df = obs_df[["id", "protocol_arm"]].drop_duplicates()
    obs_data.update({nb_patients: {"obs_df": obs_df, "patients_df": patients_df}})

In [ ]:
def train_model(input_df: pd.DataFrame) -> GP:
    descriptors = ["R0", "k_eL", "Vc"]
    gp = GP(
        input_df,
        descriptors=descriptors + ["time"],
        deep_kernel=False,
        nb_features=5,
        nb_inducing_points=100,
        min_delta=0.001,
        nb_training_iter=200,
        log_inputs=descriptors,
        log_outputs=[],
        learning_rate=0.15,
        lr_decay=0.995,
        plot_frame_rate=10000,
    )
    gp.train()

    return gp

In [ ]:
%%capture output
%timeit -n 1 -r 5 -v time_train gp = train_model(train_df)

In [ ]:
print(time_train.timings)

In [ ]:
gp = train_model(train_df)
gp.plot_obs_vs_predicted("training", (5, 5), logScale=[False])

In [ ]:
structural_gp = StructuralGp(gp)
# NLME model parameters
init_log_MI = {}
init_log_PDU = {
    "k_eL": {"mean": 0.0, "sd": 0.5},
    "R0": {"mean": 0.0, "sd": 0.5},
    "Vc": {"mean": 0.0, "sd": 0.5},
}

error_model_type = "additive"
init_res_var = [0.5]


def fit_saem(nb_patients: int) -> NlmeModel:
    obs_df, patients_df = (
        obs_data[nb_patients]["obs_df"],
        obs_data[nb_patients]["patients_df"],
    )
    nlme_surrogate = NlmeModel(
        structural_gp,
        patients_df,
        init_log_MI,
        init_log_PDU,
        init_res_var,
        None,
        error_model_type,
        num_chains=1,
        # constraints=structural_gp.training_ranges,
    )

    optimizer = PySaem(
        nlme_surrogate,
        obs_df,
        nb_phase1_iterations=100,
        nb_phase2_iterations=100,
        mcmc_nb_transitions=1,
        mcmc_first_burn_in=0,
        verbose=False,
        live_plot=False,
    )
    optimizer.run()

    return nlme_surrogate

In [ ]:
%%capture output
time_saem_all = {}
for nb in patient_counts:
    %timeit -n 1 -r 5 -v time_saem fit_saem(nb)
    time_saem_all.update({nb:time_saem.timings})

In [ ]:
perf_df_list = []
for nb in patient_counts:
    temp_df = pd.DataFrame(
        {
            "nb_patients": nb,
            "time_saem": time_saem_all[nb],
            "time_train": time_train.average,
        }
    )
    perf_df_list.append(temp_df)
perf_df = pd.concat(perf_df_list)
perf_df["time"] = perf_df["time_saem"] + perf_df["time_train"]
perf_df.to_csv("./outputs/performance_pysaem.csv", index=False)